# Basel Traffic-Light Backtest

Runs `market_risk.traffic_light_backtesting` -- the Basel Committee's 250-trading-day VaR-breach test (CLAUDE.md section 4.3: green < 5 breaches, yellow 5-9, red >= 10) -- against a synthetic 250-day return/VaR-estimate series.

Requires a free-tier API key: https://www.pyvar.com#get-api-key

In [ ]:
%load_ext pyvar_jupyter
%pyvar_key eyJ...  # replace with your own key, or set PYVAR_API_KEY before starting the kernel

## A 250-day return series with a few deliberate breaches

`traffic_light_backtesting` needs two parallel series: the actual realized returns, and the VaR estimate that was in force on each of those days. Real usage would pull both from your own risk system's history; this notebook fabricates a series with 3 *deliberately* forced breaches, holding the VaR estimate constant for simplicity -- the random baseline itself also produces a few breaches by chance (a return more than ~2.3 std devs from the mean happens roughly 1% of the time per day, so a handful over 250 days is expected, not a bug), so the actual total ends up a bit higher than 3.

In [ ]:
import random

random.seed(11)
n_days = 250
daily_var_estimate = 0.025  # held constant for this example

actual_returns = [random.gauss(0.0002, 0.011) for _ in range(n_days)]
# Force exactly 3 breaches (a loss deeper than the VaR estimate) -- green zone.
for day in (40, 120, 210):
    actual_returns[day] = -(daily_var_estimate + 0.005)

var_estimates = [daily_var_estimate] * n_days

## Run the backtest

`{actual_returns}` and `{var_estimates}` interpolate the two Python variables defined above -- the line-magic form, same variable-expansion mechanism as `01_portfolio_var.ipynb`'s `%pyvar var.compute` example, just with two substitutions instead of one.

In [ ]:
result = %pyvar market_risk.traffic_light_backtesting actual_returns={actual_returns} var_estimates={var_estimates}
result

With this seed, that comes out to `n_breaches: 6`, `basel_zone: "yellow"` -- the 3 forced breaches plus a few more from the random baseline alone (see the note above). Yellow zone (5-9 breaches) is where the BCBS capital add-on multiplier schedule kicks in above the green-zone 3.0x floor (CLAUDE.md section 4.3) -- worth pairing this call with `kupiec_pof_test`/`christoffersen_independence_test` to see whether the extra breaches look like genuine model miscalibration or a run of bad luck.